<a href="https://colab.research.google.com/github/sayam-h069/RL/blob/main/RL_Lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile grid_env_det.py
# grid_env_det.py
# Environment definition for 2x3 Grid World MDP

class GridWorldMDP:
    def __init__(self):
        # State Space (X is an obstacle and not a valid state)
        self.states = ["S0", "S1", "G", "S3", "S4"]

        self.state_coords = {
            "S0": (0, 0),
            "S1": (0, 1),
            "G": (0, 2),
            "S3": (1, 1),
            "S4": (1, 2)
        }

        self.actions = ["Up", "Down", "Left", "Right"]
        self.gamma = 0.9
        self.goal_state = "G"

        # Deterministic Transition Table
        self.transition_table = {
            "S0": {"Up": "S0", "Down": "S0", "Left": "S0", "Right": "S1"},
            "S1": {"Up": "S1", "Down": "S3", "Left": "S0", "Right": "G"},
            "S3": {"Up": "S1", "Down": "S3", "Left": "S3", "Right": "S4"},
            "S4": {"Up": "G",  "Down": "S4", "Left": "S3", "Right": "S4"},
            "G":  {"Up": "G",  "Down": "G",  "Left": "G",  "Right": "G"}
        }

    def get_reward(self, current_state, action, next_state):
        if current_state == self.goal_state:
            return 0  # Terminal
        if next_state == self.goal_state:
            return 10  # Goal reward
        return -1  # Normal movement reward

    def transition(self, state, action):
        if state not in self.states:
            raise ValueError(f"State '{state}' is invalid.")
        if action not in self.actions:
            raise ValueError(f"Action '{action}' is invalid.")
        return self.transition_table[state][action]

Writing grid_env_det.py


In [2]:
# assignment8.py
# Program to calculate State Value and Action Value by following paths

from grid_env_det import GridWorldMDP


def state_value_function(env, policy, start_state):
    """Calculates the State Value V(s) by traversing trajectory under policy without Bellman equations."""
    if start_state == env.goal_state:
        return 0.0

    current_state = start_state
    rewards = []
    max_steps = 100
    steps = 0

    while current_state != env.goal_state and steps < max_steps:
        action = policy[current_state]
        next_state = env.transition(current_state, action)
        reward = env.get_reward(current_state, action, next_state)
        rewards.append(reward)
        current_state = next_state
        steps += 1

    # Calculate discounted return G0
    total_return = sum((env.gamma**t) * r for t, r in enumerate(rewards))
    return total_return


def action_value_function(env, policy, start_state, initial_action):
    """Calculates Action Value Q(s, a) by taking initial_action first, then following policy pi."""
    # Step 1: Take specified initial action
    next_state = env.transition(start_state, initial_action)
    initial_reward = env.get_reward(start_state, initial_action, next_state)

    rewards = [initial_reward]

    # Step 2: Follow policy pi for subsequent steps
    current_state = next_state
    max_steps = 100
    steps = 0

    while current_state != env.goal_state and steps < max_steps:
        action = policy[current_state]
        next_state_p = env.transition(current_state, action)
        reward = env.get_reward(current_state, action, next_state_p)
        rewards.append(reward)
        current_state = next_state_p
        steps += 1

    # Calculate discounted return
    total_return = sum((env.gamma**t) * r for t, r in enumerate(rewards))
    return total_return


def main():
    # 1. Load MDP Module from Assignment 7
    env = GridWorldMDP()

    # 2. Define Policy pi(s) = [S0: Right, S1: Down, S3: Right, S4: Up]
    policy = {"S0": "Right", "S1": "Down", "S3": "Right", "S4": "Up", "G": "Up"}

    print("==================================================")
    print("      ASSIGNMENT 8: STATE & ACTION VALUES         ")
    print("==================================================")

    # 3. Calculate and display State Value for state 'S0'
    v_S0 = state_value_function(env, policy, "S0")
    print(f"\n1. State Value V_pi('S0') = {v_S0:.4f}")

    # 4. Calculate and display Action Value for state 'S0' and action 'Up'
    q_S0_up = action_value_function(env, policy, "S0", "Up")
    print(f"2. Action Value Q_pi('S0', 'Up') = {q_S0_up:.4f}")

    # Display values for all states and actions
    print("\n--------------------------------------------------")
    print("  State Values V(s) for All States:")
    print("--------------------------------------------------")
    for s in env.states:
        v_val = state_value_function(env, policy, s)
        print(f"  V_pi('{s}') = {v_val:.4f}")

    print("\n--------------------------------------------------")
    print("  Action Values Q(S0, a) for All Actions in S0:")
    print("--------------------------------------------------")
    for act in env.actions:
        q_val = action_value_function(env, policy, "S0", act)
        print(f"  Q_pi('S0', '{act:5s}') = {q_val:.4f}")


if __name__ == "__main__":
    main()

      ASSIGNMENT 8: STATE & ACTION VALUES         

1. State Value V_pi('S0') = 4.5800
2. Action Value Q_pi('S0', 'Up') = 3.1220

--------------------------------------------------
  State Values V(s) for All States:
--------------------------------------------------
  V_pi('S0') = 4.5800
  V_pi('S1') = 6.2000
  V_pi('G') = 0.0000
  V_pi('S3') = 8.0000
  V_pi('S4') = 10.0000

--------------------------------------------------
  Action Values Q(S0, a) for All Actions in S0:
--------------------------------------------------
  Q_pi('S0', 'Up   ') = 3.1220
  Q_pi('S0', 'Down ') = 3.1220
  Q_pi('S0', 'Left ') = 3.1220
  Q_pi('S0', 'Right') = 4.5800
